# Daten laden

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier # added (from sklearn v. 1.7)
import tpqoa
from datetime import datetime, timezone, timedelta
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

In [4]:
data = pd.read_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20260114_five_minute.csv', parse_dates = ["time"], index_col = "time")

# data = pd.read_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_seconds.csv', parse_dates = ["time"], index_col = "time")

In [5]:
data["returns"] = np.log(data.div(data.shift(1)))

In [6]:
data

,price,returns
time,,
2025-06-01 21:00:00,1.13491,NaN
2025-06-01 21:05:00,1.13470,-0.000185
2025-06-01 21:10:00,1.13491,0.000185
2025-06-01 21:15:00,1.13490,-0.000009
2025-06-01 21:20:00,1.13488,-0.000018
...,...,...
2026-01-13 23:35:00,1.16450,0.000000
2026-01-13 23:40:00,1.16434,-0.000137
2026-01-13 23:45:00,1.16436,0.000017


In [7]:
data.dropna(inplace = True)

In [8]:
data

,price,returns
time,,
2025-06-01 21:05:00,1.13470,-0.000185
2025-06-01 21:10:00,1.13491,0.000185
2025-06-01 21:15:00,1.13490,-0.000009
2025-06-01 21:20:00,1.13488,-0.000018
2025-06-01 21:25:00,1.13488,0.000000
...,...,...
2026-01-13 23:35:00,1.16450,0.000000
2026-01-13 23:40:00,1.16434,-0.000137
2026-01-13 23:45:00,1.16436,0.000017


In [9]:
data["direction"] = np.sign(data.returns)

In [10]:
data

,price,returns,direction
time,,,
2025-06-01 21:05:00,1.13470,-0.000185,-1.0
2025-06-01 21:10:00,1.13491,0.000185,1.0
2025-06-01 21:15:00,1.13490,-0.000009,-1.0
2025-06-01 21:20:00,1.13488,-0.000018,-1.0
2025-06-01 21:25:00,1.13488,0.000000,0.0
...,...,...,...
2026-01-13 23:35:00,1.16450,0.000000,0.0
2026-01-13 23:40:00,1.16434,-0.000137,-1.0
2026-01-13 23:45:00,1.16436,0.000017,1.0


In [11]:
#lags = 2
# lags= 5   # Standard
lags=10

In [12]:
cols = []
for lag in range(1, lags + 1):
    col = f'lag{lag}'
    data[col] = data.returns.shift(lag)
    cols.append(col)
data.dropna(inplace = True)

In [13]:
means = data[cols].mean()
means

# Ausgabe lag1: -0,0000003117767
# Ausgabe lag2: -0,0000003115345

lag1     5.510996e-07
lag2     5.541100e-07
lag3     5.548843e-07
lag4     5.574832e-07
lag5     5.590126e-07
lag6     5.582673e-07
lag7     5.578850e-07
lag8     5.588117e-07
lag9     5.618952e-07
lag10    5.565758e-07
dtype: float64

In [14]:
data

,price,returns,direction,lag1,lag2,lag3,lag4,lag5,lag6,lag7,lag8,lag9,lag10
time,,,,,,,,,,,,,
2025-06-01 21:55:00,1.13509,-0.000079,-1.0,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018,-0.000009,0.000185,-0.000185
2025-06-01 22:00:00,1.13520,0.000097,1.0,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018,-0.000009,0.000185
2025-06-01 22:05:00,1.13557,0.000326,1.0,0.000097,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018,-0.000009
2025-06-01 22:10:00,1.13566,0.000079,1.0,0.000326,0.000097,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018
2025-06-01 22:15:00,1.13574,0.000070,1.0,0.000079,0.000326,0.000097,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-13 23:35:00,1.16450,0.000000,0.0,0.000034,0.000000,-0.000052,0.000043,0.000060,0.000017,-0.000077,0.000000,0.000077,0.000086
2026-01-13 23:40:00,1.16434,-0.000137,-1.0,0.000000,0.000034,0.000000,-0.000052,0.000043,0.000060,0.000017,-0.000077,0.000000,0.000077
2026-01-13 23:45:00,1.16436,0.000017,1.0,-0.000137,0.000000,0.000034,0.000000,-0.000052,0.000043,0.000060,0.000017,-0.000077,0.000000


In [15]:
data[cols]

,lag1,lag2,lag3,lag4,lag5,lag6,lag7,lag8,lag9,lag10
time,,,,,,,,,,
2025-06-01 21:55:00,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018,-0.000009,0.000185,-0.000185
2025-06-01 22:00:00,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018,-0.000009,0.000185
2025-06-01 22:05:00,0.000097,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018,-0.000009
2025-06-01 22:10:00,0.000326,0.000097,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000,-0.000018
2025-06-01 22:15:00,0.000079,0.000326,0.000097,-0.000079,0.000106,0.000053,0.000053,-0.000018,0.000070,0.000000
...,...,...,...,...,...,...,...,...,...,...
2026-01-13 23:35:00,0.000034,0.000000,-0.000052,0.000043,0.000060,0.000017,-0.000077,0.000000,0.000077,0.000086
2026-01-13 23:40:00,0.000000,0.000034,0.000000,-0.000052,0.000043,0.000060,0.000017,-0.000077,0.000000,0.000077
2026-01-13 23:45:00,-0.000137,0.000000,0.000034,0.000000,-0.000052,0.000043,0.000060,0.000017,-0.000077,0.000000


In [16]:
stand_devs = data[cols].std()
stand_devs

lag1     0.000241
lag2     0.000241
lag3     0.000241
lag4     0.000241
lag5     0.000241
lag6     0.000241
lag7     0.000241
lag8     0.000241
lag9     0.000241
lag10    0.000241
dtype: float64

In [17]:
data[cols] = (data[cols]-means) / stand_devs
data

,price,returns,direction,lag1,lag2,lag3,lag4,lag5,lag6,lag7,lag8,lag9,lag10
time,,,,,,,,,,,,,
2025-06-01 21:55:00,1.13509,-0.000079,-1.0,0.435964,0.216845,0.216853,-0.075364,0.289900,-0.002314,-0.075369,-0.038844,0.764816,-0.769448
2025-06-01 22:00:00,1.13520,0.000097,1.0,-0.330966,0.435951,0.216841,0.216843,-0.075370,0.289904,-0.002313,-0.075373,-0.038857,0.764834
2025-06-01 22:05:00,1.13557,0.000326,1.0,0.399434,-0.330979,0.435948,0.216831,0.216836,-0.075367,0.289905,-0.002317,-0.075385,-0.038835
2025-06-01 22:10:00,1.13566,0.000079,1.0,1.348665,0.399422,-0.330982,0.435939,0.216825,0.216839,-0.075365,0.289901,-0.002329,-0.075363
2025-06-01 22:15:00,1.13574,0.000070,1.0,0.326258,1.348654,0.399418,-0.330994,0.435932,0.216828,0.216841,-0.075369,0.289887,-0.002307
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-13 23:35:00,1.16450,0.000000,0.0,0.140115,-0.002297,-0.215898,0.175687,0.246892,0.068891,-0.322728,-0.002317,0.318084,0.353735
2026-01-13 23:40:00,1.16434,-0.000137,-1.0,-0.002285,0.140103,-0.002300,-0.215909,0.175680,0.246895,0.068893,-0.322732,-0.002329,0.318104
2026-01-13 23:45:00,1.16436,0.000017,1.0,-0.571912,-0.002297,0.140099,-0.002311,-0.215916,0.175683,0.246897,0.068889,-0.322743,-0.002307


In [18]:
# lm = LogisticRegression(C = 1e6, max_iter = 100000, multi_class = "ovr") # old
lm = OneVsRestClassifier(LogisticRegression(C=1e6, max_iter=100000))  # new (from sklearn v. 1.7)

In [19]:
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [20]:
lm.fit(data[cols], data.direction)

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [21]:
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [22]:
data["pred"] = lm.predict(data[cols])

In [23]:
data

,price,returns,direction,lag1,lag2,lag3,lag4,lag5,lag6,lag7,lag8,lag9,lag10,pred
time,,,,,,,,,,,,,,
2025-06-01 21:55:00,1.13509,-0.000079,-1.0,0.435964,0.216845,0.216853,-0.075364,0.289900,-0.002314,-0.075369,-0.038844,0.764816,-0.769448,-1.0
2025-06-01 22:00:00,1.13520,0.000097,1.0,-0.330966,0.435951,0.216841,0.216843,-0.075370,0.289904,-0.002313,-0.075373,-0.038857,0.764834,1.0
2025-06-01 22:05:00,1.13557,0.000326,1.0,0.399434,-0.330979,0.435948,0.216831,0.216836,-0.075367,0.289905,-0.002317,-0.075385,-0.038835,1.0
2025-06-01 22:10:00,1.13566,0.000079,1.0,1.348665,0.399422,-0.330982,0.435939,0.216825,0.216839,-0.075365,0.289901,-0.002329,-0.075363,-1.0
2025-06-01 22:15:00,1.13574,0.000070,1.0,0.326258,1.348654,0.399418,-0.330994,0.435932,0.216828,0.216841,-0.075369,0.289887,-0.002307,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-13 23:35:00,1.16450,0.000000,0.0,0.140115,-0.002297,-0.215898,0.175687,0.246892,0.068891,-0.322728,-0.002317,0.318084,0.353735,-1.0
2026-01-13 23:40:00,1.16434,-0.000137,-1.0,-0.002285,0.140103,-0.002300,-0.215909,0.175680,0.246895,0.068893,-0.322732,-0.002329,0.318104,1.0
2026-01-13 23:45:00,1.16436,0.000017,1.0,-0.571912,-0.002297,0.140099,-0.002311,-0.215916,0.175683,0.246897,0.068889,-0.322743,-0.002307,1.0


In [24]:
hits = np.sign(data.direction * data.pred).value_counts()

In [25]:
hits

 1.0    22778
-1.0    21864
 0.0     1448
Name: count, dtype: int64

In [26]:
hit_ratio = hits[1.0] / sum(hits)
hit_ratio

np.float64(0.49420698633109134)

In [27]:
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [28]:
import pickle

In [29]:
pickle.dump(lm, open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20260114_logreg.pkl', "wb"))

In [30]:
params = {"mu":means, "std":stand_devs}
params

{'mu': lag1     5.510996e-07
 lag2     5.541100e-07
 lag3     5.548843e-07
 lag4     5.574832e-07
 lag5     5.590126e-07
 lag6     5.582673e-07
 lag7     5.578850e-07
 lag8     5.588117e-07
 lag9     5.618952e-07
 lag10    5.565758e-07
 dtype: float64,
 'std': lag1     0.000241
 lag2     0.000241
 lag3     0.000241
 lag4     0.000241
 lag5     0.000241
 lag6     0.000241
 lag7     0.000241
 lag8     0.000241
 lag9     0.000241
 lag10    0.000241
 dtype: float64}

In [31]:
pickle.dump(params, open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20260114_params.pkl', "wb"))

In [32]:
class MLTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, lags, model, units):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.data = None
        self.last_bar = None
        self.units = units
        self.position = 0
        self.profits = []

        #*****************add strategy-specific attributes here******************
        self.lags = lags
        self.model = model
        #************************************************************************

    def get_most_recent(self, days = 5):
        while True:
            time.sleep(2)
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            now = now - timedelta(microseconds = now.microsecond)
            past = now - timedelta(days = days)
            df = self.get_history(instrument = self.instrument, start = past, end = now,
                                   granularity = "S5", price = "M", localize = False).c.dropna().to_frame()
            df.rename(columns = {"c":self.instrument}, inplace = True)
            df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]
            self.raw_data = df.copy()
            self.last_bar = self.raw_data.index[-1]
            if pd.to_datetime(datetime.now(timezone.utc)) - self.last_bar < self.bar_length:
                break

    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")

        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        if recent_tick - self.last_bar > self.bar_length:
            self.resample_and_join()
            self.define_strategy()
            self.execute_trades()

    def resample_and_join(self):
        self.raw_data = pd.concat([self.raw_data, self.tick_data.resample(self.bar_length,
                                                                          label="right").last().ffill().iloc[:-1]])
        self.tick_data = self.tick_data.iloc[-1:]
        self.last_bar = self.raw_data.index[-1]

    def define_strategy(self): # "strategy-specific"
        df = self.raw_data.copy()

        #******************** define your strategy here ************************
        df = pd.concat([df, self.tick_data]) # new with pd.concat
        df["returns"] = np.log(df[self.instrument] / df[self.instrument].shift())
        cols = []
        for lag in range(1, self.lags + 1):
            col = f'lag{lag}'
            df[col] = df.returns.shift(lag)
            cols.append(col)
        df.dropna(inplace = True)

        df[cols] = (df[cols] - means) / stand_devs # newly added (scaling)

        df["position"] = lm.predict(df[cols])
        #***********************************************************************

        self.data = df.copy()

    def execute_trades(self):
        if self.data["position"].iloc[-1] == 1:
            if self.position == 0:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")
            elif self.position == -1:
                order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")
            self.position = 1
        elif self.data["position"].iloc[-1] == -1:
            if self.position == 0:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")
            self.position = -1
        elif self.data["position"].iloc[-1] == 0:
            if self.position == -1:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")
            self.position = 0

    def report_trade(self, order, going):
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print(f'{time} | {going}')
        print(f'{time} | units = {units} | price = {price} | P&L = {pl} | Cum P&L = {cumpl}')
        print(100 * "-" + "\n")

In [33]:
lm = pickle.load(open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20260114_logreg.pkl', "rb"))
lm

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [34]:
params = pickle.load(open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20260114_params.pkl', "rb"))
params

{'mu': lag1     5.510996e-07
 lag2     5.541100e-07
 lag3     5.548843e-07
 lag4     5.574832e-07
 lag5     5.590126e-07
 lag6     5.582673e-07
 lag7     5.578850e-07
 lag8     5.588117e-07
 lag9     5.618952e-07
 lag10    5.565758e-07
 dtype: float64,
 'std': lag1     0.000241
 lag2     0.000241
 lag3     0.000241
 lag4     0.000241
 lag5     0.000241
 lag6     0.000241
 lag7     0.000241
 lag8     0.000241
 lag9     0.000241
 lag10    0.000241
 dtype: float64}

In [35]:
means = params["mu"]
stand_devs = params["std"]

In [36]:
means

lag1     5.510996e-07
lag2     5.541100e-07
lag3     5.548843e-07
lag4     5.574832e-07
lag5     5.590126e-07
lag6     5.582673e-07
lag7     5.578850e-07
lag8     5.588117e-07
lag9     5.618952e-07
lag10    5.565758e-07
dtype: float64

In [37]:
stand_devs

lag1     0.000241
lag2     0.000241
lag3     0.000241
lag4     0.000241
lag5     0.000241
lag6     0.000241
lag7     0.000241
lag8     0.000241
lag9     0.000241
lag10    0.000241
dtype: float64

In [38]:
trader = MLTrader(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\oanda.cfg', "EUR_USD", "S5", lags = 2, model = lm, units = 10000)

In [39]:
trader.model

,estimator,LogisticRegre...x_iter=100000)
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1000000.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [40]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop = 300)
if trader.position != 0: # if we have a final open position
    close_order = trader.create_order(trader.instrument, units = -trader.position * trader.units,
                                      suppress = True, ret = True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position = 0

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 
----------------------------------------------------------------------------------------------------
2025-12-17T12:15:02.915654690Z | GOING SHORT
2025-12-17T12:15:02.915654690Z | units = -10000.0 | price = 1.17154 | P&L = 0.0 | Cum P&L = 0.0
----------------------------------------------------------------------------------------------------

50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191

In [95]:
trader.data.tail(10)

,EUR_USD,returns,lag1,lag2,position
2025-12-17 10:25:00+00:00,1.171500,-0.000051,-0.319242,-1.074444,1.0
2025-12-17 10:30:00+00:00,1.171540,0.000034,-0.173981,-0.319243,1.0
2025-12-17 10:35:00+00:00,1.171740,0.000171,0.116588,-0.173982,1.0
2025-12-17 10:40:00+00:00,1.171710,-0.000026,0.581441,0.116587,-1.0
2025-12-17 10:45:00+00:00,1.171740,0.000026,-0.086796,0.581440,-1.0
2025-12-17 10:50:00+00:00,1.172080,0.000290,0.087516,-0.086797,1.0
2025-12-17 10:55:00+00:00,1.172400,0.000273,0.987970,0.087515,-1.0
2025-12-17 11:00:00+00:00,1.172510,0.000094,0.929614,0.987969,-1.0
2025-12-17 11:05:00+00:00,1.172385,-0.000107,0.319732,0.929613,-1.0
2025-12-17 11:05:02.030385711+00:00,1.172380,-0.000004,-0.362566,0.319731,1.0


In [96]:
trader.tick_data

,EUR_USD
2025-12-17 11:05:02.030385711+00:00,1.172380
2025-12-17 11:05:02.474713396+00:00,1.172380
2025-12-17 11:05:03.010623494+00:00,1.172380
2025-12-17 11:05:05.017600781+00:00,1.172380
2025-12-17 11:05:06.095221719+00:00,1.172390
...,...
2025-12-17 11:07:06.081560777+00:00,1.172340
2025-12-17 11:07:09.170358023+00:00,1.172325
2025-12-17 11:07:10.107773340+00:00,1.172325
2025-12-17 11:07:10.222884767+00:00,1.172315
